# 06 — Sentiment Classifier Evaluation

**Phase 4 deliverable (part 2).**

Evaluates the 3-class sentiment classifier (pessimistic / neutral / optimistic)
trained on Gemini-bootstrapped labels.

## Goals
- Bootstrap 500 labels via Gemini API; manually review 100
- Train classifier on frozen BERTimbau embeddings
- Report per-class and macro F1
- Target: macro F1 > 0.65
- Visualise sentiment distribution across companies and periods

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
from pathlib import Path

from src import config

# Load sentiment labels
labels_path = config.SENTIMENT_LABELS_PATH
if not labels_path.exists():
    print(f"Labels not found at {labels_path}")
    print("Run:  python -m src.nlp.train_sentiment --bootstrap")
else:
    labeled = json.loads(labels_path.read_text())
    print(f"Loaded {len(labeled)} labeled chunks")
    dist = Counter(r["label"] for r in labeled)
    print(f"\nLabel distribution:")
    for cls in ["pessimistic", "neutral", "optimistic"]:
        n = dist.get(cls, 0)
        print(f"  {cls:12s}: {n:4d}  ({n/len(labeled)*100:.1f}%)")


Loaded 500 labeled chunks

Label distribution:
  pessimistic :   11  (2.2%)
  neutral     :  379  (75.8%)
  optimistic  :  110  (22.0%)


## 1. Label Distribution

In [2]:
dist_df = pd.DataFrame([
    {"label": cls, "count": dist.get(cls, 0)}
    for cls in ["pessimistic", "neutral", "optimistic"]
])

fig = px.bar(
    dist_df, x="label", y="count",
    color="label",
    color_discrete_map={"pessimistic": "#d62728", "neutral": "#aec7e8", "optimistic": "#2ca02c"},
    title="Sentiment Label Distribution (Gemini-bootstrapped)",
    labels={"label": "Sentiment", "count": "Count"},
)
fig.show()


## 2. Train & Evaluate Classifier

In [3]:
from sklearn.model_selection import train_test_split
from src.nlp.embedder import embed_chunks, load_model
from src.nlp.train_sentiment import train_classifier, evaluate_classifier, SENTIMENT_CLASSES
import json as _json

model_dir = config.MODELS_DIR / "sentence_transformer"

texts = [r["text"] for r in labeled]
raw_labels = [r["label"] for r in labeled]

# Check if binary mode is saved from CLI run; otherwise use 3-class
mode_path = config.SENTIMENT_MODEL_DIR / "mode.json"
binary = mode_path.exists() and _json.loads(mode_path.read_text()).get("mode") == "binary"

if binary:
    labels = ["positive" if l == "optimistic" else "negative" for l in raw_labels]
    print(f"Binary mode: positive={labels.count('positive')}  negative={labels.count('negative')}")
else:
    labels = raw_labels
    print(f"3-class mode: {dict(Counter(labels))}")

print(f"\nEmbedding {len(texts)} chunks (using {'fine-tuned' if model_dir.exists() else 'base'} model)…")
# Use CPU to avoid OOM if GPU is occupied
embeddings = embed_chunks(texts, model_dir=model_dir if model_dir.exists() else None,
                          batch_size=config.EMBEDDING_BATCH_SIZE, device="cpu")
print(f"Embedding shape: {embeddings.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=42, stratify=labels,
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

train_classifier(X_train, y_train, config.SENTIMENT_MODEL_DIR)
if binary:
    (config.SENTIMENT_MODEL_DIR / "mode.json").write_text(
        _json.dumps({"mode": "binary", "classes": ["negative", "positive"]})
    )

metrics = evaluate_classifier(config.SENTIMENT_MODEL_DIR, X_test, y_test)
print(f"\nMacro F1: {metrics['macro_f1']:.3f}  (target: ≥ 0.65)")
for cls in (["negative", "positive"] if binary else SENTIMENT_CLASSES):
    key = f"{cls}_f1"
    if key in metrics:
        print(f"  {cls:12s}: F1 = {metrics[key]:.3f}")


Binary mode: positive=110  negative=390

Embedding 500 chunks (using fine-tuned model)…


/home/conderafael/Programing/Portfolio/cvm-intelligence/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15143.27it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

...

Batches:  88%|████████▊ | 14/16 [01:01<00:08,  4.45s/it]

Batches:  94%|█████████▍| 15/16 [01:05<00:04,  4.47s/it]

Batches: 100%|██████████| 16/16 [01:08<00:00,  3.94s/it]

Batches: 100%|██████████| 16/16 [01:08<00:00,  4.28s/it]

Embedding shape: (500, 768)
Train: 400  Test: 100

Macro F1: 0.601  (target: ≥ 0.65)


## 3. Confusion Matrix

In [4]:
import joblib
from sklearn.metrics import confusion_matrix

clf = joblib.load(config.SENTIMENT_MODEL_DIR / "classifier.joblib")
le  = joblib.load(config.SENTIMENT_MODEL_DIR / "label_encoder.joblib")

y_true = le.transform(y_test)
y_pred = clf.predict(X_test)

cm = confusion_matrix(y_true, y_pred)
class_names = list(le.classes_)

fig = px.imshow(
    cm,
    x=class_names, y=class_names,
    color_continuous_scale="Blues",
    text_auto=True,
    title="Confusion Matrix — Sentiment Classifier (Test Set)",
    labels={"x": "Predicted", "y": "True"},
)
fig.update_layout(coloraxis_showscale=False)
fig.show()


## 4. Sample Predictions

In [5]:
import random as _rng

# Find the test-set texts
test_indices = list(range(len(y_test)))
_rng.seed(7)
sample_idx = _rng.sample(test_indices, min(6, len(test_indices)))

print("Sample predictions vs ground truth:\n")
for i in sample_idx:
    true_label = y_test[i]
    pred_label = le.inverse_transform([clf.predict(X_test[i:i+1])[0]])[0]
    match = "✓" if true_label == pred_label else "✗"
    # Find original text
    orig_text = next((r["text"] for r in labeled if r["label"] == true_label), "")[:150]
    print(f"  [{match}] true={true_label:12s}  pred={pred_label:12s}")
    print(f"       text: {orig_text!r}…\n")


Sample predictions vs ground truth:

  [✓] true=negative      pred=negative    
       text: ''…

  [✓] true=negative      pred=negative    
       text: ''…

  [✗] true=negative      pred=positive    
       text: ''…

  [✓] true=negative      pred=negative    
       text: ''…

  [✓] true=negative      pred=negative    
       text: ''…

  [✓] true=positive      pred=positive    
       text: ''…

